In [1]:
# Ô lệnh do kaggle_run.sh chèn vào — đặt chế độ chạy trước khi notebook đọc biến này.
import os
os.environ['SMOKE_TEST'] = '0'
print('SMOKE_TEST =', os.environ['SMOKE_TEST'], '->', 'thử nhanh' if os.environ['SMOKE_TEST'] == '1' else 'chạy đầy đủ')


SMOKE_TEST = 0 -> chạy đầy đủ


# Lazada Uplift — 04a · Huấn luyện 6 mô hình

Notebook đầu tiên trong ba mảnh tách ra từ `04_benchmark_evaluation`. Nó chỉ làm **một** việc: huấn luyện 6 mô hình CATE trên Train + Val với bộ siêu tham số notebook 03 đã chốt, rồi chấm điểm CATE trên `rct_select`.

### Vì sao tách làm ba

Chạy đủ dữ liệu thì riêng `CausalForestDML` đã ăn hết ngân sách thời gian của một kernel Kaggle. Tách ra thì mỗi mảnh có quota riêng, và hỏng chỗ nào chỉ phải chạy lại chỗ đó.

Nhưng lý do quan trọng hơn nằm ở chỗ khác: **`rct_holdout` chỉ tồn tại trong 04c.** Notebook này và 04b thậm chí không nạp file đó, nên chuyện "lỡ nhìn holdout sớm" trở thành bất khả thi về mặt vật lý chứ không còn là một lời hứa trong markdown.

```
04a_train    huấn luyện 6 mô hình            -> 04a_model_*.pkl, 04a_cate_select.npz
04b_select   chọn quán quân, chọn đặc trưng  -> decisions_locked.json, 04b_model_ban_giao.pkl
04c_holdout  mở holdout một lần, bàn giao    -> model.pkl, metadata.json, contract, golden
```

### Notebook này KHÔNG làm gì

- Không chấm Qini, không xếp hạng, không chọn quán quân — việc đó của 04b.
- Không nạp `rct_holdout.parquet`.

### Đầu ra

| File | Nội dung |
|---|---|
| `artifacts/04a_model_<tên>.pkl` | 6 mô hình đã huấn luyện |
| `artifacts/04a_cate_select.npz` | CATE dự đoán trên `rct_select` của từng mô hình |
| `artifacts/04a_train_meta.json` | `run_id`, cấu hình, số dòng huấn luyện, thời gian, lỗi (nếu có) |

---
# Phần 1 — Thiết lập

### Bước 1 — Import thư viện

In [2]:
import os
if os.path.exists('/kaggle'):
    get_ipython().run_line_magic('pip', 'install -q econml optuna mlflow')
else:
    print('Local: dependencies are managed by uv (uv sync --frozen).')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3

In [3]:
import os
import json
import time
import pickle
import hashlib
import uuid
import cloudpickle
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from econml.dml import LinearDML, NonParamDML, CausalForestDML

warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("📌 Đang chạy trên Kaggle Environment")
    # Không gõ cứng đường dẫn: Kaggle mount dataset ở chỗ khác nhau tuỳ cách gắn.
    # Tìm thẳng thư mục nào chứa train.parquet.
    import glob
    _ung_vien = [os.path.dirname(f) for f in
                 glob.glob('/kaggle/input/**/train.parquet', recursive=True)]
    if not _ung_vien:
        raise FileNotFoundError(
            'Không tìm thấy train.parquet trong /kaggle/input. Kiểm tra đã gắn dataset chưa.')
    DATA_DIR = _ung_vien[0]
    ART_DIR = '/kaggle/working/artifacts'
    FIG_DIR = '/kaggle/working/reports/figures'
else:
    print("📌 Đang chạy trên máy Local")
    ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
    DATA_DIR = os.path.join(ROOT, 'dataset')
    ART_DIR = os.path.join(ROOT, 'artifacts')
    FIG_DIR = os.path.join(ROOT, 'reports', 'figures')

os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# np.trapz đã đổi tên ở NumPy 2 — dùng bản nào có sẵn
TRAPZ = getattr(np, 'trapezoid', None) or np.trapz

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print(f"DATA_DIR: {DATA_DIR}")
print(f"ART_DIR : {ART_DIR}")

📌 Đang chạy trên Kaggle Environment
DATA_DIR: /kaggle/input/datasets/hatrungkienhatuki/lazada-uplift-data
ART_DIR : /kaggle/working/artifacts


In [4]:
import glob


def tim_file(ten, bat_buoc=True):
    """Tìm một file bàn giao: ưu tiên artifacts local, rồi tới output kernel khác trên Kaggle.

    Trên Kaggle, đầu ra của notebook trước được gắn vào qua kernel_sources và nằm
    dưới /kaggle/input/<slug>/. Tìm thấy nhiều bản thì in hết ra — hai bản khác nhau
    nghĩa là đang có bản cũ lẫn vào, và chọn nhầm bản cũ là lỗi không báo gì cả.
    """
    ung_vien = [os.path.join(ART_DIR, ten)]
    ung_vien += sorted(glob.glob(f'/kaggle/input/**/{ten}', recursive=True))
    ung_vien = [p for p in ung_vien if os.path.exists(p)]
    if not ung_vien:
        if bat_buoc:
            raise FileNotFoundError(
                f'Không tìm thấy {ten}. Chạy notebook sinh ra nó trước, hoặc gắn output '
                'của kernel đó vào qua kernel_sources.')
        return None
    if len(ung_vien) > 1:
        print(f'!!! Tìm thấy {len(ung_vien)} bản {ten} — kiểm tra có bản cũ lẫn vào không:')
        for p in ung_vien:
            print('   ', p)
    return ung_vien[0]


def bam_dac_trung(feats):
    """Vân tay của tập đặc trưng — ba notebook phải khớp nhau."""
    return hashlib.sha256(','.join(feats).encode()).hexdigest()[:16]


print('Đã định nghĩa tim_file và bam_dac_trung')

Đã định nghĩa tim_file và bam_dac_trung


### Bước 2 — Chế độ chạy

Mặc định chạy nhanh; đổi bằng biến môi trường chứ không sửa notebook.

```bash
SMOKE_TEST=0 jupyter lab
```

Trên Kaggle, đặt biến này ngay ô đầu tiên của kernel:

```python
%env SMOKE_TEST=0
```

`N_FIT` là số dòng tối đa mỗi mô hình được học. Đặt `None` nghĩa là dùng toàn bộ Train + Val.

> **Cảnh báo về tính công bằng của bảng benchmark.** Nếu các mô hình học trên số dòng khác nhau thì cột Qini ở 04b trộn hai thứ: chất lượng phương pháp và lượng dữ liệu. Muốn so sánh phương pháp cho đúng thì **cả sáu phải cùng một `N`**. Cấu hình mặc định dưới đây cho cả sáu dùng toàn bộ dữ liệu — công bằng, nhưng `CausalForestDML` sẽ là mô hình tốn thời gian nhất. Nếu kernel không kịp, hạ **cả sáu** xuống cùng một mức thay vì chỉ hạ riêng CausalForest.

In [5]:
# Mặc định chạy nhanh. Đổi bằng biến môi trường, không sửa notebook:
#     SMOKE_TEST=0  ->  chế độ đầy đủ
SMOKE_TEST = os.environ.get('SMOKE_TEST', '1').lower() not in ('0', 'false', 'no')

MODELS = ['SLearner', 'TLearner', 'DRLearner',
          'LinearDML', 'NonParamDML', 'CausalForestDML']

if SMOKE_TEST:
    N_FIT = {m: 20_000 for m in MODELS}
else:
    # Cả sáu cùng một cỡ mẫu -> bảng benchmark ở 04b so sánh được ngang hàng.
    # None = toàn bộ Train + Val (926.669 dòng).
    N_FIT = {m: None for m in MODELS}

RUN_MODE = 'smoke_test' if SMOKE_TEST else 'full'

# Vân tay của lần chạy này. 04b và 04c đối chiếu lại để không ghép nhầm kết quả
# của hai lần chạy khác nhau.
RUN_ID = f"{RUN_MODE}-{datetime.now():%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"

cung_co_mau = len(set(map(str, N_FIT.values()))) == 1

print('=' * 66)
print(f'  04a — HUẤN LUYỆN   |   CHẾ ĐỘ: {RUN_MODE.upper()}')
print(f'  run_id: {RUN_ID}')
for k, v in N_FIT.items():
    print(f'    {k:<18} {"toàn bộ Train + Val" if v is None else f"{v:,} dòng"}')
print(f'  Cùng cỡ mẫu cho cả 6 mô hình: {"CÓ" if cung_co_mau else "KHÔNG"}')
if not cung_co_mau:
    print('    >>> Bảng benchmark ở 04b sẽ KHÔNG so sánh ngang hàng được — phải nói rõ khi báo cáo <<<')
if SMOKE_TEST:
    print('  >>> KẾT QUẢ CHỈ ĐỂ KIỂM TRA CODE, KHÔNG DÙNG BÁO CÁO <<<')
print('=' * 66)

  04a — HUẤN LUYỆN   |   CHẾ ĐỘ: FULL
  run_id: full-20260819-024811-f35fc7
    SLearner           toàn bộ Train + Val
    TLearner           toàn bộ Train + Val
    DRLearner          toàn bộ Train + Val
    LinearDML          toàn bộ Train + Val
    NonParamDML        toàn bộ Train + Val
    CausalForestDML    toàn bộ Train + Val
  Cùng cỡ mẫu cho cả 6 mô hình: CÓ


### Bước 3 — Nạp tham số tốt nhất từ notebook 03

Hai thứ phải kiểm ngay:

1. **`run_mode`** — nếu notebook 03 chạy ở chế độ smoke test thì tham số trong file là tham số của một
   cuộc tìm kiếm hai trial, không dùng để báo cáo được.
2. **`xu_ly_overlap.propensity_alpha`** — ngưỡng cắt propensity mà notebook 03 đã chốt theo quy tắc
   Crump. Notebook này **phải dùng lại đúng ngưỡng đó** trong `compute_dr_scores`, nếu không thì bộ
   siêu tham số tinh chỉnh cho một mô hình lại đem áp cho một mô hình khác. Ô lệnh dưới đây đọc nó ra
   biến `E_CLIP` và Bước 15 dùng lại.

In [6]:
# best_params.json là ĐẦU RA của notebook 03, nằm ở đâu tuỳ cách chạy:
#   - local  : artifacts/best_params.json
#   - Kaggle : output của kernel notebook 03, mount trong /kaggle/input/
# Thứ tự ưu tiên đặt local trước, rồi mới quét /kaggle/input. Nếu tìm thấy nhiều hơn
# một file thì in hết ra — hai bản khác nhau nghĩa là đang có bản cũ lẫn vào, và
# chọn nhầm bản cũ là lỗi không báo gì cả.
bp_path = tim_file('best_params.json')

print('Đọc tham số từ:', bp_path)
with open(bp_path, encoding='utf-8') as f:
    tuning = json.load(f)

BEST = tuning['best_params']

print(f"run_mode của notebook 03 : {tuning['run_mode']}")
if tuning['run_mode'] != 'full':
    print('  >>> CẢNH BÁO: tham số đến từ smoke test, không dùng cho báo cáo <<<')

# --- ngưỡng cắt propensity: lấy từ notebook 03, không gõ tay lại --------------
ov = tuning.get('xu_ly_overlap')
if ov is None:
    raise KeyError(
        "best_params.json thiếu khối 'xu_ly_overlap'. File này đến từ bản notebook 03 cũ "
        "(hàm mục tiêu DR-MSE, cắt propensity ở 0,01). Chạy lại notebook 03 trước.")

E_ALPHA = float(ov['propensity_alpha'])
E_CLIP = (E_ALPHA, 1 - E_ALPHA)

print(f"Hàm mục tiêu tinh chỉnh  : {tuning['ham_muc_tieu']['ten']} "
      f"({tuning['ham_muc_tieu']['huong']})")
print(f"Ngưỡng cắt propensity    : {E_CLIP}  <- Bước 15 dùng lại đúng ngưỡng này")
print(f"Estimand của thước đo 03 : {ov['estimand']}")
print(f"Val giữ lại sau trim     : {ov['ti_le_val_giu_lai']*100:.1f}%")

fail = [m for m in MODELS if not BEST.get(m, {}).get('thanh_cong')]
print(f"\nSố mô hình có tham số dùng được : {len(MODELS) - len(fail)}/{len(MODELS)}")
if fail:
    print('  >>> Thiếu:', fail, '— quay lại notebook 03 <<<')


def _ktc(m):
    ci = BEST[m].get('dr_qini_ci')
    return '—' if not ci else f'[{ci[0]:+.4f} – {ci[1]:+.4f}]'


pd.DataFrame([{'Mô hình': m,
               'Qini_DR': BEST[m]['dr_qini'],
               'KTC 95% (Val)': _ktc(m),
               'DR-MSE': BEST[m]['dr_mse'],
               'Số trial': BEST[m]['n_trials'],
               'Tham số': ', '.join(f'{k}={v}' for k, v in BEST[m]['params'].items())}
              for m in MODELS])

Đọc tham số từ: /kaggle/input/notebooks/hatrungkienhatuki/lazada-03-tuning/artifacts/best_params.json
run_mode của notebook 03 : full
Hàm mục tiêu tinh chỉnh  : dr_qini (maximize)
Ngưỡng cắt propensity    : (0.071, 0.929)  <- Bước 15 dùng lại đúng ngưỡng này
Estimand của thước đo 03 : ATE trên vùng overlap: E_Pval[tau(X) | alpha <= e(X) <= 1-alpha]. KHÔNG phải ATE trên toàn quần thể Val.
Val giữ lại sau trim     : 43.4%

Số mô hình có tham số dùng được : 6/6


,Mô hình,Qini_DR,KTC 95% (Val),DR-MSE,Số trial,Tham số
0,SLearner,0.072704,[+0.0513 – +0.0943],0.040539,12,"n_estimators=100, learning_rate=0.026777926034..."
1,TLearner,0.032424,[+0.0115 – +0.0536],0.041442,12,"n_estimators=200, learning_rate=0.131258303162..."
2,DRLearner,0.064088,[+0.0441 – +0.0861],0.040857,12,"n_estimators=200, learning_rate=0.041415361105..."
3,LinearDML,0.067536,[+0.0487 – +0.0875],0.040484,12,"nuisance_n_estimators=300, nuisance_max_depth=..."
4,NonParamDML,0.052029,[+0.0323 – +0.0745],0.040516,12,"n_estimators=100, learning_rate=0.017001754132..."
5,CausalForestDML,0.057165,[+0.0372 – +0.0775],0.040487,6,"nuisance_n_estimators=200, nuisance_max_depth=..."


Bảng trên đến từ notebook 03 và chỉ có **một** nhiệm vụ: cho biết mỗi mô hình đã được tinh chỉnh
bằng bộ tham số nào. Đọc thêm gì nữa là sai phạm vi, vì hai lý do:

- **Qini_DR đo trên vùng overlap của Val**, một quần thể hẹp hơn hẳn và khác hẳn quần thể RCT.
  Cột `KTC 95% (Val)` cho thấy độ rộng của khoảng tin cậy ngay trên chính tập đó.
- **Cột `DR-MSE` để nguyên làm tang chứng.** Sáu con số nằm gọn trong một khoảng rất hẹp và mốc dự
  đoán hằng số (`hieu_chinh_tren_val.dr_mse_du_doan_hang_so` trong file) nằm ngay giữa khoảng đó.
  Đó chính là lý do notebook 03 phải bỏ DR-MSE làm hàm mục tiêu: nó bị phương sai của pseudo-outcome
  nuốt mất phần tín hiệu.

Việc xếp hạng giữa các mô hình thuộc về hệ số Qini đo trên tập RCT ở Phần 4, và bắt buộc phải kèm
khoảng tin cậy bootstrap.

### Bước 4 — Nạp dữ liệu

Nạp `train`, `val` và `rct_select`. **Không** nạp `rct_holdout` — file đó chỉ được mở ở 04c.

In [7]:
info_path = os.path.join(ART_DIR, 'feature_info.json')
if not os.path.exists(info_path):
    info_path = os.path.join(DATA_DIR, 'feature_info.json')

with open(info_path, encoding='utf-8') as f:
    finfo = json.load(f)

FEATS = finfo['tat_ca_dac_trung']
N_FEAT = len(FEATS)

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
val = pd.read_parquet(os.path.join(DATA_DIR, 'val.parquet'))
rct_sel = pd.read_parquet(os.path.join(DATA_DIR, 'rct_select.parquet'))

# Chốt cổng: ba nơi phải cùng nói về một tập đặc trưng — feature_info.json (notebook 02),
# best_params.json (notebook 03) và các file parquet. Lệch một cột là mọi con số phía dưới sai.
assert N_FEAT == 69, f'feature_info.json khai {N_FEAT} đặc trưng, phải là 69'
assert not any(c.startswith('fe_') for c in FEATS), 'Còn sót đặc trưng dẫn xuất fe_*'
assert finfo.get('dac_trung_dan_xuat') == []
assert tuning['cau_hinh']['feature_names'] == FEATS, \
    'Notebook 03 tinh chỉnh trên một tập đặc trưng khác — chạy lại notebook 03'
for ten, d in [('train', train), ('val', val), ('rct_select', rct_sel)]:
    assert list(d.columns) == ['data_id', 'is_treat', 'label'] + FEATS, f'{ten}: sai thứ tự cột'

print(f'Train      : {len(train):>8,} dòng | treat {train.is_treat.mean()*100:5.2f}% '
      f'| mua {train.label.mean()*100:.2f}%')
print(f'Val        : {len(val):>8,} dòng | treat {val.is_treat.mean()*100:5.2f}% '
      f'| mua {val.label.mean()*100:.2f}%')
print(f'RCT-select : {len(rct_sel):>8,} dòng | treat {rct_sel.is_treat.mean()*100:5.2f}% '
      f'| mua {rct_sel.label.mean()*100:.2f}%')
print(f'\nĐặc trưng  : {N_FEAT} cột gốc (không có đặc trưng dẫn xuất)')
print('RCT-holdout: chưa nạp — chỉ mở ở Phần 6')

Train      :  741,335 dòng | treat 22.17% | mua 1.99%
Val        :  185,334 dòng | treat 22.17% | mua 1.99%
RCT-select :   90,834 dòng | treat 52.12% | mua 3.52%

Đặc trưng  : 69 cột gốc (không có đặc trưng dẫn xuất)
RCT-holdout: chưa nạp — chỉ mở ở Phần 6


---
# Phần 2 — Sáu mô hình

### Bước 5 — Định nghĩa lại 6 mô hình

Chép nguyên từ notebook 03. Các notebook cố tình không import lẫn nhau — mỗi cái phải chạy độc lập
từ trên xuống — nên phải lặp lại phần định nghĩa. Đổi lại, nếu sau này ai chỉ mở notebook 04 thì vẫn
đọc hiểu được toàn bộ mô hình mà không phải mở file khác.

Hai điểm khác notebook 03, cả hai đều có lý do:

- **Ngưỡng cắt propensity không gõ lại.** `compute_dr_scores` mặc định dùng `E_CLIP`, biến mà Bước 3
  vừa đọc từ `best_params.json`. Gõ lại một con số ở đây là mở đường cho việc hai notebook cắt `ê` ở
  hai ngưỡng khác nhau, và khi đó bộ siêu tham số tinh chỉnh cho một mô hình lại đem áp cho một mô
  hình khác. Lưu ý notebook 03 còn **trim** vùng ngoài overlap, nhưng chỉ trim ở phía *thước đo*;
  phía huấn luyện thì cả hai notebook đều chỉ clip, nên phần này khớp nhau.
- **DR-Learner được đặt `importance_type='gain'`.** Mặc định của LightGBM là đếm **số lần** một đặc
  trưng được dùng để chia nhánh, cách đó thiên vị nặng cho đặc trưng liên tục nhiều giá trị vì chúng
  có nhiều điểm cắt để thử. Gần một nửa số cột ở đây là nhị phân, dùng mặc định sẽ đánh giá thấp
  chúng một cách oan uổng. `gain` đo mức giảm hàm mất mát thực tế mà mỗi lần chia mang lại, công bằng
  giữa hai loại.

In [8]:
def compute_dr_scores(X, W, Y, n_folds=5, seed=SEED, clip=None):
    '''Pseudo-outcome doubly robust, ước lượng bằng cross-fitting.
       clip = None nghĩa là dùng E_CLIP đọc từ best_params.json ở Bước 3 — phải đúng
       ngưỡng notebook 03 đã tinh chỉnh, nếu không thì tham số áp cho một mô hình khác.'''
    clip = E_CLIP if clip is None else clip
    n = len(X)
    e_hat = np.zeros(n)
    mu0_hat = np.zeros(n)
    mu1_hat = np.zeros(n)

    strata = W * 2 + Y
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    nuisance_params = dict(n_estimators=200, learning_rate=0.05, max_depth=5,
                           min_child_samples=50, random_state=seed, verbose=-1)

    for tr, te in skf.split(X, strata):
        m_e = lgb.LGBMClassifier(**nuisance_params).fit(X[tr], W[tr])
        e_hat[te] = np.clip(m_e.predict_proba(X[te])[:, 1], *clip)

        m0 = lgb.LGBMClassifier(**nuisance_params).fit(X[tr][W[tr] == 0], Y[tr][W[tr] == 0])
        m1 = lgb.LGBMClassifier(**nuisance_params).fit(X[tr][W[tr] == 1], Y[tr][W[tr] == 1])
        mu0_hat[te] = m0.predict_proba(X[te])[:, 1]
        mu1_hat[te] = m1.predict_proba(X[te])[:, 1]

    dr = ((mu1_hat - mu0_hat)
          + W * (Y - mu1_hat) / e_hat
          - (1 - W) * (Y - mu0_hat) / (1 - e_hat))

    return dr, e_hat, mu0_hat, mu1_hat


class SLearner:
    '''Một mô hình duy nhất trên [X, W]. CATE = f(X,1) - f(X,0).'''
    def __init__(self, **params):
        self.params = dict(params, random_state=SEED, verbose=-1)

    def fit(self, X, W, Y):
        self.model = lgb.LGBMClassifier(**self.params).fit(np.column_stack([X, W]), Y)
        return self

    def predict_cate(self, X):
        n = len(X)
        p1 = self.model.predict_proba(np.column_stack([X, np.ones(n)]))[:, 1]
        p0 = self.model.predict_proba(np.column_stack([X, np.zeros(n)]))[:, 1]
        return p1 - p0


class TLearner:
    '''Hai mô hình riêng cho nhóm treatment và control. CATE = f1(X) - f0(X).'''
    def __init__(self, **params):
        self.params = dict(params, random_state=SEED, verbose=-1)

    def fit(self, X, W, Y):
        self.m1 = lgb.LGBMClassifier(**self.params).fit(X[W == 1], Y[W == 1])
        self.m0 = lgb.LGBMClassifier(**self.params).fit(X[W == 0], Y[W == 0])
        return self

    def predict_cate(self, X):
        return self.m1.predict_proba(X)[:, 1] - self.m0.predict_proba(X)[:, 1]


class DRLearner:
    '''Kennedy 2020 — hồi quy pseudo-outcome doubly robust.'''
    def __init__(self, n_folds=5, **params):
        self.n_folds = n_folds
        self.params = dict(params, random_state=SEED, verbose=-1, importance_type='gain')

    def fit(self, X, W, Y):
        dr_train, *_ = compute_dr_scores(X, W, Y, n_folds=self.n_folds)
        self.model = lgb.LGBMRegressor(**self.params).fit(X, dr_train)
        return self

    def predict_cate(self, X):
        return self.model.predict(X)


class EconMLWrapper:
    '''Bọc estimator của EconML về cùng giao diện fit/predict_cate.'''
    def __init__(self, estimator):
        self.est = estimator

    def fit(self, X, W, Y):
        self.est.fit(Y, W, X=X)
        return self

    def predict_cate(self, X):
        return np.asarray(self.est.effect(X)).ravel()


print('Đã định nghĩa 6 lớp mô hình')

Đã định nghĩa 6 lớp mô hình


### Bước 6 — Hàm dựng mô hình từ tên và tham số

In [9]:
def make_nuisance(p):
    common = dict(n_estimators=p['nuisance_n_estimators'], learning_rate=0.05,
                  max_depth=p['nuisance_max_depth'], random_state=SEED, verbose=-1)
    return lgb.LGBMRegressor(**common), lgb.LGBMClassifier(**common)


def build_model(name, p):
    if name == 'SLearner':
        return SLearner(n_estimators=p['n_estimators'], learning_rate=p['learning_rate'],
                        max_depth=p['max_depth'], min_child_samples=p['min_child_samples'])
    if name == 'TLearner':
        return TLearner(n_estimators=p['n_estimators'], learning_rate=p['learning_rate'],
                        max_depth=p['max_depth'], min_child_samples=p['min_child_samples'])
    if name == 'DRLearner':
        return DRLearner(n_estimators=p['n_estimators'], learning_rate=p['learning_rate'],
                         max_depth=p['max_depth'], min_child_samples=p['min_child_samples'])

    model_y, model_t = make_nuisance(p)
    if name == 'LinearDML':
        return EconMLWrapper(LinearDML(
            model_y=model_y, model_t=model_t, discrete_treatment=True,
            cv=p['cv'], random_state=SEED))
    if name == 'NonParamDML':
        return EconMLWrapper(NonParamDML(
            model_y=model_y, model_t=model_t,
            model_final=lgb.LGBMRegressor(
                n_estimators=p['n_estimators'], learning_rate=p['learning_rate'],
                max_depth=p['max_depth'], min_child_samples=p['min_child_samples'],
                random_state=SEED, verbose=-1),
            discrete_treatment=True, cv=p['cv'], random_state=SEED))
    if name == 'CausalForestDML':
        return EconMLWrapper(CausalForestDML(
            model_y=model_y, model_t=model_t, discrete_treatment=True,
            n_estimators=p['cf_n_estimators'], min_samples_leaf=p['cf_min_samples_leaf'],
            max_depth=p['cf_max_depth'], cv=p['cv'], random_state=SEED))

    raise ValueError(f'Không biết mô hình: {name}')


print('Đã định nghĩa build_model')

Đã định nghĩa build_model


---
# Phần 3 — Huấn luyện

### Bước 7 — Gộp Train + Val

Notebook 03 tìm siêu tham số trên mẫu con để tiết kiệm thời gian. Giờ dùng lại đúng bộ tham số đó
nhưng huấn luyện trên **Train + Val gộp lại**: đã tinh chỉnh xong thì không còn lý do giữ Val riêng,
và thêm 185 nghìn dòng là thêm dữ liệu học miễn phí.

Dùng `float32` thay vì `float64`: 926 nghìn dòng × 69 cột ở `float64` chiếm khoảng 510 MB, còn
`float32` chỉ khoảng 255 MB. LightGBM lượng tử hóa đặc trưng thành histogram 256 bin trước khi học
nên độ chính xác dôi ra của `float64` không được dùng tới.

In [10]:
full = pd.concat([train, val], ignore_index=True)

X_full = full[FEATS].values.astype(np.float32)
W_full = full['is_treat'].values
Y_full = full['label'].values

X_sel = rct_sel[FEATS].values.astype(np.float32)

print(f'Tập huấn luyện gộp : {len(full):,} dòng × {len(FEATS)} đặc trưng')
print(f'  treat {W_full.mean()*100:.2f}% | mua {Y_full.mean()*100:.2f}%')
print(f'  bộ nhớ X: {X_full.nbytes/1e6:.0f} MB')
print(f'Tập RCT-select     : {len(rct_sel):,} dòng')

Tập huấn luyện gộp : 926,669 dòng × 69 đặc trưng
  treat 22.17% | mua 1.99%
  bộ nhớ X: 256 MB
Tập RCT-select     : 90,834 dòng


### Bước 8 — Vòng huấn luyện

Chạy lâu — ở chế độ đầy đủ khoảng 40 đến 60 phút tùy máy. Mô hình nào lỗi thì ghi lại và bỏ qua chứ
không làm hỏng cả vòng, giống cách notebook 03 xử lý trial lỗi.

In [11]:
def lay_mau(n_max, seed=SEED):
    '''Lấy mẫu phân tầng theo cặp (W, Y) để giữ nguyên tỉ lệ 4 tổ hợp.'''
    if n_max is None or n_max >= len(full):
        return np.arange(len(full))
    strata = W_full * 2 + Y_full
    rng = np.random.RandomState(seed)
    idx = []
    for s in np.unique(strata):
        pos = np.where(strata == s)[0]
        k = max(1, int(round(n_max * len(pos) / len(full))))
        idx.append(rng.choice(pos, size=min(k, len(pos)), replace=False))
    return np.sort(np.concatenate(idx))


models, cate_sel, thoi_gian, n_dong_fit, loi = {}, {}, {}, {}, {}

for name in MODELS:
    if not BEST.get(name, {}).get('thanh_cong'):
        loi[name] = 'không có tham số dùng được từ notebook 03'
        print(f'{name:<18} BỎ QUA — {loi[name]}')
        continue

    idx = lay_mau(N_FIT[name])
    t0 = time.time()
    try:
        m = build_model(name, BEST[name]['params']).fit(X_full[idx], W_full[idx], Y_full[idx])
        cate = m.predict_cate(X_sel)
        if not np.all(np.isfinite(cate)):
            raise ValueError('CATE chứa giá trị không hữu hạn')
        models[name], cate_sel[name] = m, cate
        thoi_gian[name] = time.time() - t0
        n_dong_fit[name] = len(idx)
        print(f'{name:<18} {len(idx):>8,} dòng | {thoi_gian[name]:>6.1f}s | '
              f'CATE trung bình {cate.mean():+.5f} | độ lệch chuẩn {cate.std():.5f}')
    except Exception as exc:
        loi[name] = f'{type(exc).__name__}: {exc}'
        print(f'{name:<18} LỖI — {loi[name]}')

OK_MODELS = [m for m in MODELS if m in models]
print(f'\n{len(OK_MODELS)}/{len(MODELS)} mô hình huấn luyện thành công')

SLearner            926,669 dòng |   11.0s | CATE trung bình +0.02140 | độ lệch chuẩn 0.02565
TLearner            926,669 dòng |   20.8s | CATE trung bình +0.00724 | độ lệch chuẩn 0.05750
DRLearner           926,669 dòng |  213.6s | CATE trung bình +0.00667 | độ lệch chuẩn 0.02360
LinearDML           926,669 dòng |  171.7s | CATE trung bình +0.00590 | độ lệch chuẩn 0.00743
NonParamDML         926,669 dòng |   86.0s | CATE trung bình +0.00694 | độ lệch chuẩn 0.00635
CausalForestDML     926,669 dòng | 1861.6s | CATE trung bình +0.00597 | độ lệch chuẩn 0.00191

6/6 mô hình huấn luyện thành công


### Bước 9 — Phân phối CATE dự đoán

Nhìn trước khi chấm điểm. Ba dấu hiệu cần soi:

- **Độ lệch chuẩn xấp xỉ 0** — mô hình dự đoán gần như một hằng số, tức nó không tìm được tính không
  đồng nhất nào. Qini của nó sẽ quanh 0 và điều đó không có gì bất ngờ.
- **Trung bình lệch xa ATE thật của tập RCT (0,37 pp)** — dấu hiệu mô hình mang theo thiên vị chọn lọc
  từ dữ liệu quan sát sang.
- **Biên độ quá rộng** — CATE là hiệu hai xác suất nên về mặt toán học phải nằm trong [-1, 1]; nếu nó
  trải rộng hơn hẳn thang của ATE thì phần lớn biên độ đó là nhiễu, không phải tín hiệu.

In [12]:
y_sel = rct_sel['label'].values
w_sel = rct_sel['is_treat'].values

bang_cate = pd.DataFrame([{
    'Mô hình': m,
    'Trung bình': cate_sel[m].mean(),
    'Độ lệch chuẩn': cate_sel[m].std(),
    'Nhỏ nhất': cate_sel[m].min(),
    'Phân vị 25%': np.percentile(cate_sel[m], 25),
    'Trung vị': np.median(cate_sel[m]),
    'Phân vị 75%': np.percentile(cate_sel[m], 75),
    'Lớn nhất': cate_sel[m].max(),
} for m in OK_MODELS])

ate_sel = (y_sel[w_sel == 1].mean() - y_sel[w_sel == 0].mean())
print(f'ATE thật đo trên rct_select: {ate_sel:.5f}  ({ate_sel*100:.3f} pp)\n')
display(bang_cate.round(5))

ATE thật đo trên rct_select: 0.00372  (0.372 pp)



,Mô hình,Trung bình,Độ lệch chuẩn,Nhỏ nhất,Phân vị 25%,Trung vị,Phân vị 75%,Lớn nhất
0,SLearner,0.02140,0.02565,-0.00731,0.00887,0.01304,0.02244,0.23306
1,TLearner,0.00724,0.05750,-0.75151,0.00167,0.00530,0.01523,0.52907
2,DRLearner,0.00667,0.02360,-1.09362,0.00328,0.00470,0.00863,0.58285
3,LinearDML,0.00590,0.00743,-0.03715,0.00036,0.00515,0.01079,0.06972
4,NonParamDML,0.00694,0.00635,-0.12390,0.00416,0.00555,0.00803,0.17083
5,CausalForestDML,0.00597,0.00191,-0.00031,0.00457,0.00560,0.00690,0.01997


---
# Phần 4 — Bàn giao cho 04b

### Bước 10 — Ghi mô hình, dự đoán và siêu dữ liệu

Ba thứ được ghi ra, và 04b cần cả ba:

- **Mô hình đã huấn luyện** — 04b phải dùng lại chính chúng cho permutation importance và cho bản bàn giao, không được huấn luyện lại một bản khác.
- **CATE trên `rct_select`** — nguyên liệu để 04b xếp hạng.
- **`04a_train_meta.json`** — `run_id`, cấu hình, số dòng thực tế mỗi mô hình học, thời gian chạy. Số dòng phải đi kèm kết quả, vì mô hình học trên ít dữ liệu hơn thì không so ngang hàng được.

Mô hình được đóng gói bằng `cloudpickle` để class định nghĩa trong notebook vẫn nạp lại được ở kernel khác.

In [13]:
os.makedirs(ART_DIR, exist_ok=True)

# --- mô hình ---
for name in OK_MODELS:
    p = os.path.join(ART_DIR, f'04a_model_{name}.pkl')
    with open(p, 'wb') as f:
        cloudpickle.dump(models[name], f)
    print(f'{p} — {os.path.getsize(p)/1e6:.2f} MB')

# --- CATE trên rct_select ---
cate_path = os.path.join(ART_DIR, '04a_cate_select.npz')
np.savez_compressed(cate_path, **{k: v for k, v in cate_sel.items()})
print(f'\n{cate_path} — {len(cate_sel)} mảng × {len(rct_sel):,} dòng')

# --- siêu dữ liệu ---
train_meta = {
    'run_id': RUN_ID,
    'run_mode': RUN_MODE,
    'seed': SEED,
    'ngay_chay': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'nguon_tham_so': {'file': 'best_params.json', 'run_mode': tuning['run_mode']},
    'dac_trung': FEATS,
    'so_dac_trung': N_FEAT,
    'ma_bam_dac_trung': bam_dac_trung(FEATS),
    'du_lieu_huan_luyen': {'nguon': 'train.parquet + val.parquet', 'so_dong_co_the_dung': int(len(full))},
    'cau_hinh_n_fit': {k: (None if v is None else int(v)) for k, v in N_FIT.items()},
    'cung_co_mau': bool(cung_co_mau),
    'mo_hinh': {
        name: {
            'so_dong_huan_luyen': int(n_dong_fit[name]),
            'giay': round(float(thoi_gian[name]), 1),
            'sieu_tham_so': BEST[name]['params'],
            'cate_trung_binh': float(cate_sel[name].mean()),
            'cate_do_lech_chuan': float(cate_sel[name].std()),
            'file': f'04a_model_{name}.pkl',
        } for name in OK_MODELS
    },
    'mo_hinh_loi': loi,
    'ghi_chu': ('CATE trong 04a_cate_select.npz đo trên rct_select. Notebook này không '
                'chấm Qini và không nạp rct_holdout.'),
}

meta_path = os.path.join(ART_DIR, '04a_train_meta.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(train_meta, f, ensure_ascii=False, indent=2)

print(f'\n{meta_path}')
print(f'  run_id      : {RUN_ID}')
print(f'  mô hình OK  : {len(OK_MODELS)}/{len(MODELS)}')
print(f'  cùng cỡ mẫu : {"CÓ" if cung_co_mau else "KHÔNG"}')
if loi:
    print(f'  LỖI         : {loi}')

/kaggle/working/artifacts/04a_model_SLearner.pkl — 0.11 MB
/kaggle/working/artifacts/04a_model_TLearner.pkl — 1.38 MB
/kaggle/working/artifacts/04a_model_DRLearner.pkl — 0.44 MB
/kaggle/working/artifacts/04a_model_LinearDML.pkl — 6.08 MB
/kaggle/working/artifacts/04a_model_NonParamDML.pkl — 1.75 MB
/kaggle/working/artifacts/04a_model_CausalForestDML.pkl — 15.31 MB

/kaggle/working/artifacts/04a_cate_select.npz — 6 mảng × 90,834 dòng

/kaggle/working/artifacts/04a_train_meta.json
  run_id      : full-20260819-024811-f35fc7
  mô hình OK  : 6/6
  cùng cỡ mẫu : CÓ


---
## Kết luận

Xong phần tốn thời gian nhất. Từ đây trở đi không còn phải huấn luyện lại 6 mô hình nữa — 04b chỉ huấn luyện lại **một** mô hình quán quân cho vòng quét đặc trưng.

### Kiểm trước khi sang 04b

- `mo_hinh_loi` phải rỗng. Mô hình nào lỗi thì 04b sẽ bỏ nó khỏi bảng xếp hạng, và bảng benchmark thiếu một phương pháp.
- `cung_co_mau` nên là `true`. Nếu `false` thì bảng ở 04b không so sánh ngang hàng được và báo cáo phải nói rõ.
- `nguon_tham_so.run_mode` phải là `full` nếu định lấy số để báo cáo.

### Notebook tiếp theo

`04b_select_model.ipynb` — chấm Qini trên `rct_select`, chọn quán quân, đo độ quan trọng đặc trưng, quét số đặc trưng, rồi **khóa toàn bộ quyết định** trước khi ai được phép mở holdout.